In [3]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from datetime import datetime
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from scipy.stats import linregress
from pathlib import Path
from abc import ABCMeta, abstractmethod
from time import time

In [4]:
sys.path.append(os.path.abspath('..'))
from configs.config import *
# from src.util import Logger, Util

In [5]:
# import importlib
# import configs.config
# importlib.reload(configs.config)
# from configs.config import *

In [6]:
pd.set_option("display.max_columns",200)
pd.set_option("display.max_rows", 500)

# 基底クラス

In [7]:
def decorate(s: str, decoration=None):
    if decoration is None:
        decoration = '★' * 20

    return ' '.join([decoration, str(s), decoration])

class Timer:
    def __init__(self, logger=None, format_str='{:.3f}[s]', prefix=None, suffix=None, sep=' ', verbose=0):

        if prefix: format_str = str(prefix) + sep + format_str
        if suffix: format_str = format_str + sep + str(suffix)
        self.format_str = format_str
        self.logger = logger
        self.start = None
        self.end = None
        self.verbose = verbose

    @property
    def duration(self):
        if self.end is None:
            return 0
        return self.end - self.start

    def __enter__(self):
        self.start = time()

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.end = time()
        if self.verbose is None:
            return
        out_str = self.format_str.format(self.duration)
        if self.logger:
            self.logger.info(out_str)
        else:
            print(out_str)

In [8]:
class FeatureBase(metaclass=ABCMeta):

    def __init__(self, use_cache=False, save_cache=False, logger=None):
        self.use_cache = use_cache
        self.name = self.__class__.__name__
        self.cache_dir = Path(DIR_FEATURE)
        self.logger = logger
        self.seve_cache = save_cache
        self.use_cols = None
        self.key_column = None
    
    # 共通のキー整形 & 重複チェック
    def enforce_key_integrity(self, df: pd.DataFrame) -> pd.DataFrame:
        for key in self.key_column:
            if key not in df.columns:
                raise KeyError(f"{self.name}: キーカラム '{key}' が存在しません")
        assert ~df[self.key_column].duplicated().any(), f"{self.name}: 主キー {self.key_column} に重複があります"
    
    @abstractmethod
    def _create_feature(self) -> pd.DataFrame:
        """
        特徴量生成の実装をサブクラスで定義する必要があります。
        :return: pd.DataFrame 生成された特徴量
        """
        raise NotImplementedError()

    # 特徴量生成処理
    def create_feature(self) -> pd.DataFrame:

        # クラス名.pkl
        file_name = os.path.join(self.cache_dir, f"{self.name}.pkl")

        # キャッシュを使う & ファイルがあるなら読み出し
        if os.path.isfile(str(file_name)) and self.use_cache:
            feature = pd.read_pickle(file_name)

        # 変換処理を実行
        else:
            # train/testの区別なく変換処理を実行
            feature = self._create_feature()

            # 主キーチェック
            if self.key_column is not None:
                self.enforce_key_integrity(feature)

            # 保存する場合
            if self.seve_cache:
                feature.to_pickle(file_name)

        return feature

In [34]:
def one_hot_encode(df, col, drop_col=True):
    """
    特定の列に対してOne-Hotエンコーディングを適用します。
    
    :param df: pd.DataFrame 対象のDataFrame
    :param col: str エンコードする列名
    :return: pd.DataFrame エンコードされたDataFrame
    """
    # Initialize OneHotEncoder
    encoder = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')

    # Fit and transform the specified column
    encoded = encoder.fit_transform(df[[col]])

    # Convert the encoded array to a DataFrame
    encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out([col]))

    # concat
    df_concat = pd.concat([df, encoded_df], axis=1)

    # drop 
    if drop_col:
        df_concat.drop(columns=col, inplace=True)

    return df_concat

# 継承クラス

In [49]:
class Key(FeatureBase):
    """
    TrainFeatureクラスは、train.csvデータを処理し、特徴量を生成します。
    """
    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache=use_cache, save_cache=save_cache, logger=logger)
        self.key_column = ['社員番号', 'category']  # 主キーとなるカラムを定義

    def _create_feature(self) -> pd.DataFrame:
        """
        train.csvデータを読み込み、特徴量を生成します。

        Returns:
        pd.DataFrame: 生成された特徴量を含むDataFrame。
        """
        # train.csvデータを読み込む
        df_train = pd.read_pickle(os.path.join(DIR_INTERIM, 'df_prep_train.pkl'))
        df_test = pd.read_pickle(os.path.join(DIR_INTERIM, 'df_prep_test.pkl'))
        df_Key = pd.concat([df_train, df_test], ignore_index=True)[self.key_column]

        return df_Key

In [27]:
class Target(FeatureBase):
    """
    Targetクラスは、ターゲットデータを処理し、特徴量を生成します。
    """
    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache=use_cache, save_cache=save_cache, logger=logger)
        self.key_column = ['社員番号', 'category']  # 主キーとなるカラムを定義

    def _create_feature(self) -> pd.DataFrame:
        """
        ターゲットデータを読み込み、特徴量を生成します。

        Returns:
        pd.DataFrame: 生成されたターゲットデータを含むDataFrame。
        """
        # ターゲットデータを読み込む
        df_train = pd.read_pickle(os.path.join(DIR_INTERIM, 'df_prep_train.pkl'))

        # 必要なカラムを選択
        df_target = df_train[['社員番号', 'category', 'target']]

        # 主キーとターゲット列を含むDataFrameを返す
        return df_target

In [46]:
class CategoryFeature(FeatureBase):
    """
    TrainFeatureクラスは、train.csvデータを処理し、特徴量を生成します。
    """
    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache=use_cache, save_cache=save_cache, logger=logger)
        self.key_column = ['category']  # 主キーとなるカラムを定義

    def _create_feature(self) -> pd.DataFrame:
        """
        train.csvデータを読み込み、特徴量を生成します。

        Returns:
        pd.DataFrame: 生成された特徴量を含むDataFrame。
        """
        # train.csvデータを読み込む
        df_train = pd.read_pickle(os.path.join(DIR_INTERIM, 'df_prep_train.pkl'))
        df_test = pd.read_pickle(os.path.join(DIR_INTERIM, 'df_prep_test.pkl'))
        df_all = pd.concat([df_train, df_test], ignore_index=True)

        df_category_feature = df_all.copy().drop_duplicates('category')[self.key_column]

        # One-hotエンコーディング
        df_category_feature = one_hot_encode(df_category_feature, 'category', False)

        # 主キーとターゲット列を含むDataFrameを返す
        return df_category_feature

In [12]:
class CareerFeature(FeatureBase):
    """
    CareerBlockクラスは、キャリアデータを処理し、特徴量を生成します。
    """
    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache=use_cache, save_cache=save_cache, logger=logger)
        self.key_column = ['社員番号']  # 主キーとなるカラムを定義

    def _create_feature(self) -> pd.DataFrame:
        """
        キャリアデータを読み込み、特徴量を生成します。

        Returns:
        pd.DataFrame: 生成された特徴量を含むDataFrame。
        """
        # 前処理済みのキャリアデータを読み込む
        df_career = pd.read_pickle(os.path.join(DIR_INTERIM, "df_prep_career.pkl"))

        df_career_feature = df_career.copy()

        # 特徴量例: キャリア関連の質問に対するポジティブな回答数
        df_career_feature['positive_responses'] = df_career_feature.iloc[:, 1:].apply(lambda row: (row == 1).sum(), axis=1)

        # 特徴量例: キャリア関連の質問に対するネガティブな回答数
        df_career_feature['negative_responses'] = df_career_feature.iloc[:, 1:].apply(lambda row: (row == 0).sum(), axis=1)

        # 特徴量例: ポジティブな回答の割合
        df_career_feature['positive_ratio'] = df_career_feature['positive_responses'] / (df_career_feature.shape[1] - 1)

        # 主キーと新しい特徴量を含むDataFrameを返す
        return df_career_feature

In [13]:
class UdemyActivityFeature(FeatureBase):
    """
    UdemyActivityFeatureクラスは、Udemyの活動データを処理し、特徴量を生成します。
    """
    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache=use_cache, save_cache=save_cache, logger=logger)
        self.key_column = ['社員番号']  # 主キーとなるカラムを定義

    def _create_feature(self) -> pd.DataFrame:
        """
        Udemy活動データを読み込み、特徴量を生成します。

        Returns:
        pd.DataFrame: 生成された特徴量を含むDataFrame。
        """
        # 前処理済みのUdemy活動データを読み込む
        df_udemy_activity = pd.read_pickle(os.path.join(DIR_INTERIM, "df_prep_udemy_activity.pkl"))

        # 総受講数
        df_udemy_activity_feature = df_udemy_activity.groupby(self.key_column).agg(
            total_learning_count=('コースID', 'count'),
        ).reset_index()

        # 主キーと新しい特徴量を含むDataFrameを返す
        return df_udemy_activity_feature

In [14]:
class DxFeature(FeatureBase):
    """
    DxFeatureクラスは、DX関連のデータを処理し、特徴量を生成します。
    """
    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache=use_cache, save_cache=save_cache, logger=logger)
        self.key_column = ['社員番号']  # 主キーとなるカラムを定義

    def _create_feature(self) -> pd.DataFrame:
        """
        DX関連データを読み込み、特徴量を生成します。

        Returns:
        pd.DataFrame: 生成された特徴量を含むDataFrame。
        """
        # 前処理済みのDXデータを読み込む
        df_dx = pd.read_pickle(os.path.join(DIR_INTERIM, "df_prep_dx.pkl"))

        # 特徴量例: 各社員の研修参加回数
        df_dx_feature = df_dx.groupby(self.key_column).agg(
            total_training_count=('研修名', 'count'),
        ).reset_index()

        # 特徴量例: 各社員のユニークな研修カテゴリ数
        # df_dx_feature['unique_training_categories'] = dx_data.groupby(self.key_column)['研修カテゴリ'].transform('nunique')

        return df_dx_feature
    

In [15]:
class HrFeature(FeatureBase):
    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache, save_cache, logger=None)
        self.key_column = ['社員番号']

    def _create_feature(self) -> pd.DataFrame:

        df_hr = pd.read_pickle(os.path.join(DIR_INTERIM, 'df_prep_hr.pkl'))

        # 実施期間を算出（日数）
        df_hr['研修日数'] = (df_hr['実施終了日'] - df_hr['実施開始日']).dt.days + 1
        df_hr['研修日数'] = df_hr['研修日数'].fillna(1).clip(lower=1)

        # 特徴量作成
        df_hr_feature = df_hr.groupby("社員番号").agg(
            n_hr_total=("研修名", "count"),
            n_hr_unique_program=("研修名", "nunique"),
            n_hr_unique_category=("カテゴリ", "nunique"),
            first_hr_date=("実施開始日", "min"),
            last_hr_date=("実施終了日", "max"),
            n_hr_days=("研修日数", "sum"),
        ).reset_index()

        # 活動期間（最終日 - 初日）
        df_hr_feature["hr_active_days"] = (df_hr_feature["last_hr_date"] - df_hr_feature["first_hr_date"]).dt.days
        df_hr_feature.drop(["first_hr_date", "last_hr_date"], axis=1, inplace=True)

        return df_hr_feature

In [16]:
class OvertimeWorkByMonthFeature(FeatureBase):
    """
    OvertimeWorkFeatureクラスは、月ごとの残業データを処理し、特徴量を生成します。
    """
    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache=use_cache, save_cache=save_cache, logger=logger)
        self.key_column = ['社員番号']  # 主キーとなるカラムを定義

    def _create_feature(self) -> pd.DataFrame:
        """
        残業データを読み込み、特徴量を生成します。

        Returns:
        pd.DataFrame: 生成された特徴量を含むDataFrame。
        """
        # 残業データを読み込む
        df_overtime = pd.read_pickle(os.path.join(DIR_INTERIM, "df_prep_overtime_work_by_month.pkl"))

        # 特徴量例: 各社員の月ごとの平均残業時間
        df_overtime_feature = df_overtime.groupby(self.key_column).agg(
            avg_overtime_hours=('hours', 'mean'),
            max_overtime_hours=('hours', 'max'),
            min_overtime_hours=('hours', 'min'),
            total_overtime_hours=('hours', 'sum'),
        ).reset_index()

        return df_overtime_feature

In [17]:
class PositionHistoryFeature(FeatureBase):
    """
    PositionHistoryFeatureクラスは、役職履歴データを処理し、特徴量を生成します。
    """
    def __init__(self, use_cache=False, save_cache=False, logger=None):
        super().__init__(use_cache=use_cache, save_cache=save_cache, logger=logger)
        self.key_column = ['社員番号']  # 主キーとなるカラムを定義

    def _create_feature(self) -> pd.DataFrame:
        """
        役職履歴データを読み込み、特徴量を生成します。

        Returns:
        pd.DataFrame: 生成された特徴量を含むDataFrame。
        """
        # 役職履歴データを読み込む
        df_position_history = pd.read_pickle(os.path.join(DIR_INTERIM, "df_prep_position_history.pkl"))

        # 特徴量例: 各社員の役職変更回数
        df_position_history_feature = df_position_history.groupby(self.key_column).agg(
            position_change_count=('役職', 'nunique'),
            first_position=('役職', 'first'),
            last_position=('役職', 'last'),
        ).reset_index()

        # one-hotエンコーディング(OneHotEncoder)
        df_position_history_feature = one_hot_encode(df_position_history_feature, 'first_position')
        df_position_history_feature = one_hot_encode(df_position_history_feature, 'last_position')

        return df_position_history_feature

# 処理実行

In [20]:
def run_blocks(feature_blocks):
    print('start run blocks...')
    with Timer(prefix='run test'):
        for block in feature_blocks:
            with Timer(prefix='\t- {}'.format(str(block))):
                feature = block.create_feature()

In [50]:
feature_blocks = [
    Key(use_cache=False, save_cache=True, logger=None),
	Target(use_cache=False, save_cache=True, logger=None),
    CategoryFeature(use_cache=False, save_cache=True, logger=None),
	CareerFeature(use_cache=False, save_cache=True, logger=None),
	UdemyActivityFeature(use_cache=False, save_cache=True, logger=None),
	DxFeature(use_cache=False, save_cache=True, logger=None),
	HrFeature(use_cache=False, save_cache=True, logger=None),
	OvertimeWorkByMonthFeature(use_cache=False, save_cache=True, logger=None),
	PositionHistoryFeature(use_cache=False, save_cache=True, logger=None),
]

In [51]:
run_blocks(feature_blocks)

start run blocks...
	- <__main__.Key object at 0x0000017D3FB98E80> 0.100[s]
	- <__main__.Target object at 0x0000017D3FB98C70> 0.102[s]
	- <__main__.CategoryFeature object at 0x0000017D3FB981C0> 0.072[s]
	- <__main__.CareerFeature object at 0x0000017D3FB98A90> 0.083[s]
	- <__main__.UdemyActivityFeature object at 0x0000017D3FB98940> 0.620[s]
	- <__main__.DxFeature object at 0x0000017D3FB986A0> 0.056[s]
	- <__main__.HrFeature object at 0x0000017D3FB983D0> 0.059[s]
	- <__main__.OvertimeWorkByMonthFeature object at 0x0000017D3FB985B0> 0.068[s]
	- <__main__.PositionHistoryFeature object at 0x0000017D3FB989D0> 0.096[s]
run test 1.257[s]
